# Milestone 5 lab, notebook 4 of 8: Tell the person who can act

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| **4. Tell the person who can act** (you are here) | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and two minutes of waiting** while Airflow runs two nights and a test runs.

In [1]:
import subprocess
import sys

import pandas as pd
from lab_helpers import (
    MILESTONE_DIR, MON, TUE, airflow_tools, alerts_log, callback_said, run_nights, runbook,
    routing_table, show_file, shouted, sql, start_again,
)

## Step 1: A row nobody reads

Back to **Ines Marchetti** and her missing Tuesday file. In milestone 4 the pipeline knew at 02:00 on Wednesday morning that her day could not be paid from. It wrote a row in `ops.gate_results`. Nobody reads that table at 02:00, or at 09:00.

This time, the head of retail operations gets a ticket that night: "`S04` Tue 08-25: No sales file arrived. **$587.42** apart. Ask the store's back office to send the file."

An **alert** is a message to a person that something needs them. It says what failed, how bad it is, and what to do. Its opposite is the row in a table that waits for someone to look.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

## Step 2: The callback

How does Airflow know when to send one? It calls back.

A **callback** is a function that Airflow calls by itself when a task changes state, for example when it fails. You write the function. You hand it to Airflow. Airflow decides when to call it.

Here is the one in this pipeline's DAG file:

In [2]:
show_file("dags/lotus_nightly_dag.py", "def tell_someone", 8)

# dags/lotus_nightly_dag.py, from line 34
def tell_someone(context) -> None:
    """Called by Airflow when a task fails, and when publish succeeds."""
    task = context["ti"].task_id
    day = context["ds"]
    failed = context["ti"].state == "failed"
    command = f"{PIPELINE} alerts --day {day}" + (f" --failed-task {task}" if failed else "")
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    print(done.stdout or done.stderr)  # the alerts also show in this task's log


And here is where it is handed to Airflow:

In [3]:
dag_file = (MILESTONE_DIR / "dags" / "lotus_nightly_dag.py").read_text().splitlines()
print("\n".join(line for line in dag_file if "callback=" in line or "callback\"" in line))

    default_args={"retries": 0, "on_failure_callback": tell_someone},
        on_success_callback=tell_someone,


Two callbacks:

| Callback | On | Fires when | What it sends |
|---|---|---|---|
| `on_failure_callback` | Every task, through `default_args` | A gate blocks, or a step crashes | A page to the engineer on call |
| `on_success_callback` | The `publish` task | The night is published | The tickets of the night |

Both run the same plain command: `python -m lotus_pipeline alerts --day <night>`. So the work is split:

- **Airflow knows when.** A task failed, or the night was published.
- **The pipeline knows whom and what.** It reads tonight's results, decides who must be told, and writes the message.
- **dbt still holds the rules.**

Why a callback, and not a last task called "alert" at the end of the DAG? You saw it in milestone 4: when a gate fails, the tasks after it are `upstream_failed`. They never start. A last task would stay silent on exactly the night that matters. A callback still fires.

## Step 3: Watch it happen

Empty everything, and run Monday and Tuesday again. This time, read what was sent.

In [4]:
start_again()
run_nights([MON])

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.


,night,tasks,succeeded,seconds,stores that can be paid
0,Mon 08-24,8,8,19,5


In [5]:
alerts_log()

2026-08-25 02:00  TICKET  to Head of retail operations: S03 Mon 08-24: Rows were set aside, and the day does not add up. 1 row, $9.49: A line adds up; A refund gives money back. The receipts and the tills are $9.49 apart. Tomas Reyes's bonus waits.
                  Do: Look at the rows in quarantine.sales_lines. Ask the store for a corrected file. Do not pay yet.
                  (RUNBOOK.md#rows-were-set-aside)


One line, written during Monday's night by the callback on `publish`. It is a **ticket** to the head of retail operations, about `S03`'s Monday: the refund of $9.49. Now Tuesday.

In [6]:
run_nights([TUE])

,night,tasks,succeeded,seconds,stores that can be paid
0,Tue 08-25,8,8,19,4


In [7]:
alerts_log()

2026-08-25 02:00  TICKET  to Head of retail operations: S03 Mon 08-24: Rows were set aside, and the day does not add up. 1 row, $9.49: A line adds up; A refund gives money back. The receipts and the tills are $9.49 apart. Tomas Reyes's bonus waits.
                  Do: Look at the rows in quarantine.sales_lines. Ask the store for a corrected file. Do not pay yet.
                  (RUNBOOK.md#rows-were-set-aside)
2026-08-26 02:00  TICKET  to Finance, and the buying team: The file supplier_prices_SUP-B_2026-08-25.csv was refused. An expected column is missing: unit_cost. The last good file is in use.
                  Do: Ask the sender for a file that keeps the contract. Until then nothing new is used.
                  (RUNBOOK.md#a-file-was-refused)
2026-08-26 02:00  TICKET  to Head of retail operations: S04 Tue 08-25: No sales file arrived. The receipts and the tills are $587.42 apart. Ines Marchetti's bonus waits.
                  Do: Ask the store's back office to send the file.

Two new lines for Tuesday night. A supplier's file was refused, to finance and the buying team. And Ines's day, to the head of retail operations, at 02:00 on Wednesday 26 August: the night it happened, not six days later.

Monday's ticket was not sent again: it is still the same problem.

The callback also prints into the log of the task it was called for. This is the end of Tuesday's `publish` log in Airflow:

In [8]:
callback_said(TUE)

alerts:       2 sent (0 page, 2 ticket). 1 still open, not sent again. 0 resolved. 2 in the digest.
    TICKET  to Finance, and the buying team: The file supplier_prices_SUP-B_2026-08-25.csv was refused. An expected column is missing: unit_cost. The last good file is in use. Do: Ask the sender for a file that keeps the contract. Until then nothing new is used. (RUNBOOK.md#a-file-was-refused)
    TICKET  to Head of retail operations: S04 Tue 08-25: No sales file arrived. The receipts and the tills are $587.42 apart. Ines Marchetti's bonus waits. Do: Ask the store's back office to send the file. Do not pay this store's bonus yet. (RUNBOOK.md#ask-a-store-for-its-file)


**Look at it yourself:** in Airflow, click Tuesday's `publish` box, then **Logs**. The same lines are at the end.

## Step 4: What one alert says

Each alert is also a row in `ops.alerts`. Here is Ines's:

In [9]:
sql("""
    SELECT level, to_whom, title, how_bad, what_to_do, runbook
    FROM ops.alerts
    WHERE fingerprint = 'store_day|S04|2026-08-25'
""").T

,0
level,ticket
to_whom,Head of retail operations
title,S04 Tue 08-25: No sales file arrived
how_bad,The receipts and the tills are $587.42 apart. Ines Marchetti's bonus waits.
what_to_do,Ask the store's back office to send the file. Do not pay this store's bonus yet.
runbook,ask-a-store-for-its-file


| It must say | Here |
|---|---|
| **What**, in the words of the business | `S04` Tue 08-25: No sales file arrived |
| **How bad** | $587.42 apart. Ines Marchetti's bonus waits |
| **What to do** | Ask the store's back office to send the file. Do not pay yet |
| **Where to read more** | A section of the runbook |

Now the opposite. The careless alert says only which test failed:

In [10]:
print("\n".join(line for line in shouted(TUE) if "ready_to_pay" in line))

ALERT to everyone: the test store_is_not_ready_to_pay failed with 2 bad rows


Which store? Which day? How much money? Who should act? It does not say. Nobody can act on it at 02:00.

## Step 5: Routing

**Routing** is sending each alert to the person who can act on it. An alert to everyone is an alert to no one. The decisions live in one place, `lotus_pipeline/alerts.py`:

In [11]:
routing_table()

,when,level,to whom
0,a test whose action is block fails,page,The data engineer on call
1,a task crashes,page,The data engineer on call
2,a store's day cannot be paid from,ticket,Head of retail operations
3,a supplier's file is refused,ticket,"Finance, and the buying team"
4,price_of_the_day: ... charged a price that is not in the price list,ticket,Finance
5,order_without_a_count: ... has a product with no stock count,ticket,The replenishment team
6,stores_did_not_change: ... changed in the stores table,ticket,Head of retail operations
7,price_change_is_believable: ... has a new price far from its old price,ticket,The buying team
8,count_values_allowed: ... has a stock count that cannot be true. It was set aside,ticket,The replenishment team
9,cost_is_filled: ... has no cost. The row was set aside,ticket,"Finance, and the buying team"


Three levels:

- **page**: wake someone now. Only when nothing new could be published: every reader sees old numbers.
- **ticket**: today. A bonus waits, an order must be made by hand, money was lost at a till.
- **message**: when you can. It is news, not a task. It waits in a digest, a list read once a day.

The person is not always the data team. A missing stock count goes to the replenishment team, who can count a shelf. A refused supplier file goes to finance and the buying team, who talk to the supplier.

## Step 6: The runbook

At 02:00 nobody should have to think from zero. A **runbook** is a short note that says what an alert means and the first things to check. Every alert ends with a link to one section of `RUNBOOK.md`. Ines's alert points at `ask-a-store-for-its-file`:

In [12]:
runbook("Ask a store for its file")

## Ask a store for its file

**What it means.** An open store has no sales file, or no till totals, for a day. The other stores were published. This store's day is marked "do not pay yet". Level: **ticket**.

**First things to check.**

1. Was the store open? The calendar says: `SELECT * FROM clean.calendar`.
2. Is the file only late? `python -m lotus_pipeline health --day <day>` lists late files and missing files apart.
3. Did the tills send their totals? Then the store was trading, and only the export failed.

**What to do.** Ask the store's back office to send the file. When it arrives, the next night loads it by itself, and this alert is closed. Do not pay the store's bonus for that week before then.


## Step 7: Only the destination is local

Each alert goes to three places: the table `ops.alerts`, one line in the alerts log file, and the log of the Airflow task. No email is sent, and no phone rings.

A real team would plug in email, Slack, or a paging service such as PagerDuty: a service that rings the phone of the engineer on call, and calls the next person if nobody answers. Those need an account, and this project needs none. The callback is real; only the destination is local. The one line to change is where `alerts.py` writes the log:

In [13]:
show_file("lotus_pipeline/alerts.py", "    if lines:", 4)

# lotus_pipeline/alerts.py, from line 362
    if lines:
        config.ALERTS_LOG.parent.mkdir(parents=True, exist_ok=True)
        with config.ALERTS_LOG.open("a") as log:
            log.write("\n".join(lines) + "\n")


## Step 8: The page

No gate blocked this week, so no page was sent. The production tests make one happen: they delete a store from the stores table, run the gates, and check that the engineer on call is paged and the run still stops. Run that one test:

In [14]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_know_alert_recover.py", "-k", "blocked_gate_pages",
     "-q", "--no-header", "--color=no", "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
print(done.stdout.strip().splitlines()[-1])

1 passed, 33 deselected in 5.81s


## What you now know

- An **alert** is a message to a person: what failed, how bad it is, what to do. Its opposite is a row in a table that nobody reads.
- A **callback** is a function Airflow calls by itself when a task changes state. Here it fires when any task fails, and when `publish` succeeds. A callback fires even when the tasks after a failure never start.
- Airflow knows **when** to tell. The pipeline's `alerts` command knows **whom** and **what**. dbt holds the rules.
- **Routing** sends each alert to the one person who can act on it, at one of three levels: page, ticket, message.
- A **runbook** says what an alert means and the first things to check. Every alert links to one section.

**Say it in an interview:** "Alerts were sent from Airflow callbacks: one on every task for a failure, one on publish for success. The callback only decided when. A plain command decided who was told and how urgently. Ours said: store S04, Tuesday, no sales file, $587.42 at stake, ask the back office to send it, and do not pay the bonus yet. Each alert linked to its section of the runbook."

**The theory:** [Airflow: Monitoring, Logging, Alerting & Production Failures](https://www.d8loop.com/modules/knowledge-corner/airflow/monitoring-logging-alerting-and-production-failures), and [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section on alerting.

Next: [notebook 5](05_too_many_alerts.ipynb). Ten alerts in a week. Why not seventy?